# 01 — Classical OEC reference model

The quantum formulations developed in the paper are meaningful only if they can be traced back to the physical resource-allocation problem from which they originate. For this reason, the first scientific notebook implements the time-expanded Orbital Edge Computing model independently of QUBO conversion and quantum optimization, and establishes a small deterministic instance whose optimum can be interpreted directly in terms of routing, storage and processing placement.

The graph follows the two-state construction used for Earth-observation OEC: a state node identifies a physical node, a time cycle and whether the service is still unprocessed or has already been processed. Transmission edges move data between physical nodes without changing its processing state, storage edges advance the service to the following time cycle, and processing edges connect the unprocessed and processed layers at the same physical node. Shared transmission, memory and processing capacities are imposed on the corresponding physical resources, while per-node energy budgets constrain the operations performed on board.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa import canonical_c1
from oec_qaoa.ilp import ordered_service_path, solve_reference_ilp
from oec_qaoa.validation import validate_reference_solution

instance = canonical_c1()
print(instance.name)

## Canonical validation instance C1

C1 is deliberately small because its role is not to approximate a realistic constellation. It is constructed to exercise the same logical elements that will later enter the QUBO formulation while retaining an optimum that can be checked analytically.

Two services are generated at satellites (S_0) and (S_2) at time (t=0). Both must reach the virtual ground station (G) in the processed state by (t=3). The services meet at (S_1), where a single on-board processing opportunity is available at (t=1), and they share a downlink of capacity (4) MB at (t=2). The first service has sizes (4\to1) MB before and after processing, whereas the second has sizes (3\to1) MB.

The resulting capacities intentionally couple the two placement decisions. Processing both services on board would exceed the (4) MB processing capacity at (S_1); processing neither would require (7) MB on the downlink; and processing only the second service would still require (5) MB. The only feasible structural choice is therefore to process (h_0) on board and (h_1) at the virtual ground station.

In [ ]:
services = pd.DataFrame(
    [
        {
            "service": service.name,
            "source": service.source,
            "generation": service.generation_time,
            "deadline": service.deadline,
            "unprocessed_mb": service.unprocessed_mb,
            "processed_mb": service.processed_mb,
        }
        for service in instance.services
    ]
)
services

In [ ]:
edges = pd.DataFrame(
    [
        {
            "edge": edge.name,
            "kind": edge.kind,
            "source": str(edge.source),
            "target": str(edge.target),
            "resource": edge.resource_key,
            "unit_energy_j_per_mb": edge.unit_energy_j_per_mb,
        }
        for edge in instance.edges
    ]
)

edges.groupby("kind").size().rename("number_of_edges").to_frame()

The synthetic construction disables processing at (S_0) and (S_2) and exposes only the processing edge at (S_1,t=1), together with ground processing at the delivery cycle. This restriction is specific to the validation instance; the general model allows processing opportunities to be generated for any physical node and time cycle for which computing resources are available.

In [ ]:
pd.Series(
    instance.resource_capacities_mb,
    name="capacity_mb",
).rename_axis("resource").to_frame()

## Exact constrained solution

The reference formulation assigns a binary variable to each admissible service-edge pair and minimizes the energy associated with processing unprocessed data at the virtual ground station. Flow conservation connects each service source to the processed ground-state destination, while physical capacities are shared across services and, where appropriate, across the two processing layers.

The solver used here is SciPy's mixed-integer interface backed by HiGHS. It is intentionally independent of the QUBO implementation that will be introduced in the next notebook; agreement between the two formulations must therefore result from the mathematics of the encoding rather than from reusing the same conversion routine.

In [ ]:
solution = solve_reference_ilp(instance)
report = validate_reference_solution(instance, solution)

print(f"Solver success: {solution.success}")
print(f"Message: {solution.message}")
print(f"Ground-processing objective: {solution.objective:.3f} J")

In [ ]:
path_rows = []
for service in instance.services:
    path = ordered_service_path(instance, solution, service.name)
    path_rows.append(
        {
            "service": service.name,
            "processing_node": report.processing_nodes[service.name],
            "path": " -> ".join(path),
        }
    )

pd.DataFrame(path_rows)

## Regression target

With the ground-processing coefficient normalized to (1) J/MB, the expected objective is (3) J because only the (3) MB service (h_1) is processed at the ground station. More importantly, the numerical optimum must reproduce the physical structure established above: (h_0) is processed at (S_1), (h_1) is processed at (G), and the shared downlink carries (1+3=4) MB.

These checks are kept as explicit assertions because the same instance will be reused when the direct edge-QUBO is introduced. A later QUBO implementation is accepted only if it reproduces this constrained optimum after decoding.

In [ ]:
assert solution.success
assert report.objective == 3.0
assert report.processing_nodes == {"h0": "S1", "h1": "G"}
assert solution.uses("h0", "tx_S1_G_t2_p1")
assert solution.uses("h1", "tx_S1_G_t2_p0")

print("C1 reference validation: PASS")

## What this notebook establishes

C1 is not a numerical benchmark and its objective value will not be used as evidence for the performance of the proposed quantum method. It provides the first validation gate of the implementation: the time-expanded graph, the physical resource accounting and the exact constrained solver agree on a solution whose structure can be established independently.

The next stage will retain the same instance and replace the constrained binary program by the direct edge-based QUBO derived in the paper. The comparison will then test equality of the optimum together with equality of the decoded service paths and processing locations, before any variational quantum solver is introduced.